# Izdelava aplikacij za generiranje slik (Azure OpenAI)

LLM-ji niso namenjeni samo generiranju besedil. Prav tako lahko generirate slike iz besedilnih opisov. Slike kot modaliteta so uporabne na področjih MedTech, arhitekture, turizma, razvoja iger, marketinga in še več. V tej lekciji delamo z današnjimi modeli **GPT Image** na platformi Microsoft Foundry.

## Cilji učenja

Na koncu te lekcije boste znali:

- Pojasniti, kaj je generiranje slik in kje je uporabno.
- Razumeti družino modelov `gpt-image` in kako se razlikujejo od starejših modelov DALL·E.
- Izdelati aplikacijo za generiranje slik in urediti slike.

## Kaj je generiranje slik?

Modeli za generiranje slik ustvarjajo slike na podlagi besedilnega navodila. Sodobni modeli, kot je `gpt-image`, med učenjem spoznavajo povezavo med besedilom in slikami, nato pa postopoma preoblikujejo naključni šum v sliko, ki ustreza vašemu navodilu.

Dve dobro poznani družini modelov za slike sta:

- **`gpt-image` (OpenAI)** — trenutna generacija, uporabljena v tej lekciji. Podpira generiranje slike iz besedila in urejanje slik (inpainting z masko).
- **Midjourney** — priljubljen model tretje strani z lastno storitvijo in delovnim procesom na Discordu.

> Starejši OpenAI modeli za slike — **DALL·E 2** in **DALL·E 3** — so zastareli. DALL·E 3 ni več na voljo za nove namestitve, funkcije, kot je `create_variation`, so bile prisotne samo v DALL·E 2. Za nove aplikacije uporabljajte modele `gpt-image`.

Na platformi Microsoft Foundry je **`gpt-image-2`** najnovejši in najzmogljivejši model za slike ter priporočena privzeta izbira. `gpt-image-1.5` in `gpt-image-1-mini` sta tudi na voljo nasploh.

> **Pomembno:** modeli `gpt-image` vrnejo generirano sliko v obliki **base64** (`b64_json`), ne kot URL. Vaša koda dekodira niz base64 v bajte in jih shrani — ni URL-ja slike za prenos.


## Izdelava vaše prve aplikacije za generiranje slik

Kaj torej potrebujete za izdelavo aplikacije za generiranje slik? Potrebujete naslednje knjižnice:

- **python-dotenv**, zelo priporočamo, da uporabite to knjižnico za shranjevanje vaših skrivnosti v datoteko *.env* ločeno od kode.
- **openai**, ta knjižnica omogoča interakcijo z OpenAI API.
- **pillow**, za delo s slikami v Pythonu.

Če tega še niste storili, sledite navodilom na strani [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) za ustvarjanje Microsoft Foundry vira in modela. Izberite **gpt-image-2** kot model (najnovejši Azure OpenAI slikovni model; DALL·E je zastarel).

1. Ustvarite datoteko *.env* z naslednjo vsebino:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Navedene informacije poiščite v Microsoft Foundry portalu za vaš vir v razdelku "Deployments".



1. Zberite zgornje knjižnice v datoteko z imenom *requirements.txt* na naslednji način:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Nato ustvarite virtualno okolje in namestite knjižnice:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Za Windows uporabite naslednje ukaze za ustvarjanje in aktivacijo vašega virtualnega okolja:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Dodajte naslednjo kodo v datoteko z imenom *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # uvozi dotenv
    dotenv.load_dotenv()

    # konfiguriraj Azure OpenAI (Microsoft Foundry) odjemalca.
    # Modeli za slike potrebujejo najnovejšo različico API-ja — preveri dokumentacijo Foundry za različico, ki jo tvoj model zahteva.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Ustvari sliko z uporabo API-ja za generiranje slik
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Vnesi besedilo poziva tukaj
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # npr. "gpt-image-2"
        )
        # Nastavi imenik za shranjeno sliko
        image_dir = os.path.join(os.curdir, 'images')

        # Če imenik ne obstaja, ga ustvari
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Inicializiraj pot do slike (upoštevaj, da mora biti tip datoteke png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image modeli vrnejo sliko v obliki base64 (b64_json), ne URL-ja
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Prikaži sliko v privzetem pregledovalniku slik
        image = Image.open(image_path)
        image.show()

    # ujemi izjeme
    except BadRequestError as err:
        print(err)

    ```

Razložimo to kodo:

- Najprej uvozimo potrebne knjižnice, vključno s knjižnico OpenAI, knjižnico dotenv, modulom base64 in knjižnico Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Nato naložimo okoljske spremenljivke iz datoteke *.env*.

    ```python
    # uvozi dotenv
    dotenv.load_dotenv()
    ```

- Potem konfiguriramo Azure OpenAI (Microsoft Foundry) odjemalca.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Naslednji korak je generiranje slike:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Vnesite besedilo poziva tukaj
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Modeli `gpt-image` vrnejo sliko kot **base64** niz v `data[0].b64_json`. Dekodiramo ga v bajte in zapišemo v datoteko — dostopne ni URL za prenos.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Nazadnje odpremo sliko in jo prikažemo z običajnim pregledovalnikom slik:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Več podrobnosti o generiranju slike

Oglejmo si parametre funkcije `images.generate`:

- **prompt**, je besedilni poziv, ki se uporablja za generiranje slike. Tukaj je to "Zajec na konju, ki drži liziko, na megleni travi, kjer rastejo velikonočnice".
- **size**, je velikost generirane slike (na primer `1024x1024`, `1536x1024`, `1024x1536` ali `"auto"`).
- **n**, je število generiranih slik. Tukaj generiramo eno.
- **model**, je ime vaše namestitve modela za slike (na primer `gpt-image-2`).

> Modeli slik ne sprejemajo parametra `temperature` — to je kontrola za generiranje besedila. Za večjo raznolikost pokličite API večkrat; za manj raznolikosti naredite vaš poziv bolj specifičen.

## Dodatne zmogljivosti generiranja slik

Videli ste, kako generirati sliko z nekaj vrsticami Pythona. Modeli `gpt-image` lahko tudi **urejajo** obstoječo sliko. Z zagotovitvijo slike, opcijskega **maskiranja** (ki označuje področje za spremembo) in poziva, lahko spremenite del slike — na primer dodate klobuk našemu zajcu.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# spremembe so prav tako vrnjene kot base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Osnovna slika vsebuje samo zajca; končna slika ima klobuk na zajcu.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
